# RakForest — Download Raw Audio Datasets

โน้ตบุ๊กนี้ดาวน์โหลดชุดข้อมูลเสียงลงใน `data/raw/` จากเครื่องของเราโดยตรง ไม่ได้รันบน Colab

Credentials ของ Kaggle/Hugging Face อ่านจาก `.env` ที่ root ของโปรเจกต์ ส่วนโฟลเดอร์ Google Drive จะลองดาวน์โหลดด้วยลิงก์แชร์ก่อน หากโฟลเดอร์ไม่เปิดให้ทุกคนที่มีลิงก์ดูได้ จึงค่อยต้องใช้ Google ADC หรือดาวน์โหลด WAV มาวางเอง

## 1. ติดตั้ง dependencies

In [35]:
import sys
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'kaggle', 'datasets[audio]', 'python-dotenv', 'google-api-python-client', 'google-auth', 'requests'])


0

## 2. ตั้งค่าโปรเจกต์และโหลด credentials
ไฟล์ `.env` ควรมี `KAGGLE_USERNAME`, `KAGGLE_KEY` และ `HF_TOKEN`

In [36]:
from pathlib import Path
import os
from dotenv import load_dotenv
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
load_dotenv(PROJECT_ROOT / '.env')
missing = [k for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY') if not os.getenv(k)]
if missing: raise RuntimeError(f'กรุณาใส่ {missing} ใน .env ก่อนดาวน์โหลด')
print('Project root:', PROJECT_ROOT)
print('Kaggle credentials loaded:', True)
print('Hugging Face token available:', bool(os.getenv('HF_TOKEN')))

Project root: /Users/guidegiegg/Projects/RakForest-ML
Kaggle credentials loaded: True
Hugging Face token available: True


## 3. ตัวช่วยดาวน์โหลดและแตก ZIP
ใช้ข้ามไฟล์ที่มีอยู่แล้ว และดาวน์โหลดแบบ streaming เพื่อไม่โหลดไฟล์ทั้งก้อนเข้า RAM

In [37]:
import requests
import zipfile

AUDIO_EXTENSIONS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac', '.wma', '.aiff', '.aif'}

def has_audio(folder):
    folder = Path(folder)
    return folder.exists() and any(
        p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS
        for p in folder.rglob('*')
    )

def download_file(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        print(f'[skip download] {destination.name}')
        return destination
    partial = destination.with_suffix(destination.suffix + '.part')
    with requests.get(url, stream=True, timeout=(30, 120)) as response:
        response.raise_for_status()
        with partial.open('wb') as f:
            for chunk in response.iter_content(8 * 1024 * 1024):
                if chunk: f.write(chunk)
    partial.replace(destination)
    return destination

def extract_zip(archive, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    if has_audio(destination):
        print(f'[skip extract] audio already exists in {destination}')
        return
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(destination)
    print(f'[extracted] {destination}')


## 4. ESC-50 (GitHub)
ดาวน์โหลดไฟล์ ZIP จาก repository ทางการแล้วแตกไว้ใน `data/raw/esc50/`.

In [38]:
ESC50_DIR = RAW_DIR / 'esc50'
if has_audio(ESC50_DIR):
    print('[skip] ESC-50: พบไฟล์เสียงแล้ว')
else:
    esc_zip = download_file('https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip', RAW_DIR / 'archives' / 'esc50.zip')
    extract_zip(esc_zip, ESC50_DIR)


[skip] ESC-50: พบไฟล์เสียงแล้ว


## 5. Footstep-set (Zenodo)
ดาวน์โหลดชุด Footstep-set เท่านั้น (ไม่โหลด Impact-set ที่อยู่ record เดียวกัน).

In [39]:
FOOTSTEP_DIR = RAW_DIR / 'footstep_set'
if has_audio(FOOTSTEP_DIR):
    print('[skip] Footstep-set: พบไฟล์เสียงแล้ว')
else:
    foot_zip = download_file('https://zenodo.org/records/14286414/files/Footstep-set.zip?download=1', RAW_DIR / 'archives' / 'Footstep-set.zip')
    extract_zip(foot_zip, FOOTSTEP_DIR)


[skip] Footstep-set: พบไฟล์เสียงแล้ว


## 6. DataSEC v2 (Zenodo)
ไฟล์บีบอัดประมาณ 6.4 GB; ตรวจสอบพื้นที่ว่างก่อนเริ่ม.

In [40]:
DATASEC_DIR = RAW_DIR / 'datasec_v2'
LEGACY_DATASEC_DIR = RAW_DIR / 'datasec_vehicle_v2'
if not has_audio(DATASEC_DIR) and has_audio(LEGACY_DATASEC_DIR):
    DATASEC_DIR = LEGACY_DATASEC_DIR
if has_audio(DATASEC_DIR):
    print('[skip] DataSEC v2: พบไฟล์เสียงแล้ว')
else:
    datasec_zip = download_file('https://zenodo.org/records/15393250/files/DATASEC.zip?download=1', RAW_DIR / 'archives' / 'DATASEC.zip')
    extract_zip(datasec_zip, DATASEC_DIR)


[skip] DataSEC v2: พบไฟล์เสียงแล้ว


## 7. RFCx / FrugalAI (Hugging Face)
ต้องมี `HF_TOKEN` ใน `.env` และยอมรับเงื่อนไขการเข้าถึง dataset บน Hugging Face ก่อน

ถ้าเห็น `TqdmWarning: IProgress not found` ระหว่างดาวน์โหลด เป็นเพียง warning ของ progress bar; การดาวน์โหลดมักทำต่อได้ตามปกติ. มี `ipywidgets` ใน cell ติดตั้ง dependencies สำหรับการรันครั้งถัดไปแล้ว ไม่ต้องรันติดตั้งซ้ำหรือ restart kernel ระหว่าง RFCx กำลังโหลด


ดาวน์โหลดและบันทึก Hugging Face dataset ไว้ใน `data/raw/rfcx_frugalai/`.


In [41]:
from datasets import load_dataset
RFCX_DIR = RAW_DIR / 'rfcx_frugalai' / 'hf_dataset'
if RFCX_DIR.exists() and any(RFCX_DIR.iterdir()):
    print('[skip] RFCx FrugalAI: พบข้อมูลอยู่แล้ว')
else:
    RFCX_DIR.parent.mkdir(parents=True, exist_ok=True)
    rfcx = load_dataset('rfcx/frugalai', token=os.getenv('HF_TOKEN') or None, cache_dir=str(RFCX_DIR.parent / '.hf_cache'))
    rfcx.save_to_disk(str(RFCX_DIR))
    print('[done] RFCx FrugalAI:', RFCX_DIR)

[skip] RFCx FrugalAI: พบข้อมูลอยู่แล้ว


## 8. FSC22 และ Vehicle Sounds (Kaggle)
ใช้ Kaggle API credentials จาก `.env`; ดาวน์โหลดสองชุดตามรายการที่ให้มา.

In [42]:
KAGGLE_DATASETS = {
    'fsc22': ('irmiot22/fsc22-dataset', RAW_DIR / 'fsc22'),
    'vehicle_sounds': ('janboubiabderrahim/vehicle-sounds-dataset', RAW_DIR / 'vehicle_sounds'),
}
for name, (dataset_id, destination) in KAGGLE_DATASETS.items():
    destination.mkdir(parents=True, exist_ok=True)
    if has_audio(destination):
        print(f'[skip] {name}: มีไฟล์เสียงอยู่แล้ว')
        continue
    print(f'[download] {name}: {dataset_id}')
    subprocess.run([sys.executable, '-m', 'kaggle', 'datasets', 'download', '-d', dataset_id, '-p', str(destination), '--unzip'], check=True, cwd=PROJECT_ROOT)


[skip] fsc22: มีไฟล์เสียงอยู่แล้ว
[skip] vehicle_sounds: มีไฟล์เสียงอยู่แล้ว


## 9. FSD50K v1.0 (Zenodo)
ชุดนี้ใหญ่มากและประกอบด้วยไฟล์ ZIP แบบแบ่ง part; cell นี้ดึงรายการไฟล์จาก Zenodo API และดาวน์โหลดทุกไฟล์ของ record ก่อนประกอบ archive. ต้องมีพื้นที่ว่างหลายสิบ GB.

In [43]:
USE_FSD50K = False  # เปิดเป็น True หากต้องการดาวน์โหลด/แตกไฟล์ FSD50K อีกครั้ง
if not USE_FSD50K:
    print('[paused] FSD50K: ข้ามการดาวน์โหลดและแตกไฟล์')
else:
    import requests
    import subprocess
    FSD_DIR = RAW_DIR / 'fsd50k_v1'
    FSD_ARCHIVES = RAW_DIR / 'archives' / 'fsd50k_v1'
    FSD_DIR.mkdir(parents=True, exist_ok=True)
    FSD_ARCHIVES.mkdir(parents=True, exist_ok=True)
    if has_audio(FSD_DIR):
        print('[skip] FSD50K: พบไฟล์เสียงแล้ว')
    else:
        record = requests.get('https://zenodo.org/api/records/4060432', timeout=60).json()
        files = record.get('files', [])
        selected = [f for f in files if f.get('key', '').lower().endswith(('.zip', '.z01', '.z02', '.z03', '.z04', '.z05'))]
        print('Archive files:', len(selected))
        for item in selected:
            download_file(item['links']['self'], FSD_ARCHIVES / item['key'])
        # Merge split archives using the system zip utility, then extract each complete ZIP.
        for prefix in ('FSD50K.dev_audio', 'FSD50K.eval_audio'):
            parts = sorted(FSD_ARCHIVES.glob(prefix + '.z*'))
            main_zip = FSD_ARCHIVES / (prefix + '.zip')
            if main_zip.exists() and parts:
                merged = FSD_ARCHIVES / (prefix + '_merged.zip')
                subprocess.run(['zip', '-s', '0', str(main_zip), '--out', str(merged)], check=True)
                with zipfile.ZipFile(merged) as zf: zf.extractall(FSD_DIR)
            elif main_zip.exists():
                with zipfile.ZipFile(main_zip) as zf: zf.extractall(FSD_DIR)
        for key in ('FSD50K.ground_truth.zip', 'FSD50K.metadata.zip', 'FSD50K.doc.zip'):
            archive = FSD_ARCHIVES / key
            if archive.exists():
                with zipfile.ZipFile(archive) as zf: zf.extractall(FSD_DIR)
        print('[done] FSD50K:', FSD_DIR)


[paused] FSD50K: ข้ามการดาวน์โหลดและแตกไฟล์


## 10. Google Drive Fire Sounds
ลองดาวน์โหลดจากลิงก์แชร์โดยตรงก่อน หากโฟลเดอร์จำกัดสิทธิ์ โค้ดจะลองใช้ Google ADC ที่ตั้งไว้ในเครื่องเป็นทางสำรอง

หากโฟลเดอร์ Drive ตั้งเป็น “ทุกคนที่มีลิงก์ดูได้” ให้รัน cell ถัดไปได้เลย ไม่ต้องตั้งค่า ADC หากเป็นโฟลเดอร์ส่วนตัว cell จะลองใช้ ADC และแจ้งวิธีตั้งค่าเมื่อจำเป็น

In [44]:
import importlib.util
import json
import subprocess
import sys

if importlib.util.find_spec('gdown') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gdown'])
import gdown

DRIVE_FOLDER_ID = '1QOYouznSh3YZR_6tbDPDQIHNyLorzqXN'
DRIVE_DIR = RAW_DIR / 'fire_google_drive'
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_MARKER = DRIVE_DIR / '.download_complete.json'

def find_drive_wavs():
    return sorted(p for p in DRIVE_DIR.rglob('*') if p.is_file() and p.suffix.lower() == '.wav')

wav_files = find_drive_wavs()
marker_data = {}
if DOWNLOAD_MARKER.exists():
    try:
        marker_data = json.loads(DOWNLOAD_MARKER.read_text())
    except Exception:
        marker_data = {}

if marker_data.get('folder_id') == DRIVE_FOLDER_ID and marker_data.get('wav_count') == len(wav_files) and wav_files:
    print(f'[skip] ดาวน์โหลดครบแล้วตาม marker: {len(wav_files):,} WAV')
else:
    print(f'พบ WAV เดิม {len(wav_files):,} ไฟล์; จะตรวจ/ดาวน์โหลดโฟลเดอร์ที่แชร์ให้ครบก่อนทำเครื่องหมายเสร็จ')
    public_download_ok = False
    try:
        gdown.download_folder(
            id=DRIVE_FOLDER_ID,
            output=str(DRIVE_DIR),
            quiet=False,
            use_cookies=False,
        )
        public_download_ok = True
    except Exception as e:
        print(f'ดาวน์โหลดแบบลิงก์แชร์ไม่สำเร็จ: {type(e).__name__}: {e}')

    wav_files = find_drive_wavs()
    if not public_download_ok:
        # If public link download failed, try Google Application Default Credentials.
        try:
            from google.auth import default
            from googleapiclient.discovery import build
            from googleapiclient.http import MediaIoBaseDownload
            credentials, _ = default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
        except Exception as e:
            raise RuntimeError(
                'ยังดาวน์โหลด WAV ไม่ได้: โฟลเดอร์ควรแชร์เป็น Anyone with the link (Viewer) '
                'ถ้ายังเข้าไม่ได้ ให้รัน `gcloud auth application-default login '
                '--scopes=https://www.googleapis.com/auth/drive.readonly` ใน Terminal '
                'ด้วยบัญชีที่มีสิทธิ์อ่านโฟลเดอร์ แล้วรัน cell นี้ใหม่'
            ) from e

        drive = build('drive', 'v3', credentials=credentials, cache_discovery=False)
        items, page_token = [], None
        while True:
            result = drive.files().list(
                q=f"'{DRIVE_FOLDER_ID}' in parents and trashed = false",
                pageSize=1000, pageToken=page_token,
                fields='nextPageToken, files(id,name,mimeType,size)',
                includeItemsFromAllDrives=True, supportsAllDrives=True,
            ).execute()
            items.extend(result.get('files', []))
            page_token = result.get('nextPageToken')
            if not page_token:
                break
        wav_items = [f for f in items if f.get('mimeType') == 'audio/wav' or f.get('name', '').lower().endswith('.wav')]
        if not wav_items:
            raise RuntimeError('เชื่อมต่อ Drive ได้แล้ว แต่ไม่พบ WAV ในโฟลเดอร์ที่ระบุ หรือบัญชีนี้ไม่มีสิทธิ์อ่าน')
        for i, item in enumerate(wav_items, 1):
            out = DRIVE_DIR / Path(item['name']).name
            expected = int(item.get('size') or 0)
            if out.exists() and (not expected or out.stat().st_size == expected):
                continue
            request = drive.files().get_media(fileId=item['id'], supportsAllDrives=True)
            with out.open('wb') as fh:
                downloader = MediaIoBaseDownload(fh, request, chunksize=8 * 1024 * 1024)
                done = False
                while not done:
                    _, done = downloader.next_chunk()
            if i % 25 == 0 or i == len(wav_items):
                print(f'Downloaded {i}/{len(wav_items)}')
        wav_files = find_drive_wavs()

    if not wav_files:
        raise RuntimeError('ไม่พบไฟล์ WAV หลังดาวน์โหลด; ตรวจสิทธิ์และโครงสร้างโฟลเดอร์ Drive')
    DOWNLOAD_MARKER.write_text(json.dumps({'folder_id': DRIVE_FOLDER_ID, 'wav_count': len(wav_files)}, indent=2))

print(f'[done] Google Drive WAV: {len(wav_files):,} files in {DRIVE_DIR}')

[skip] ดาวน์โหลดครบแล้วตาม marker: 721 WAV
[done] Google Drive WAV: 721 files in /Users/guidegiegg/Projects/RakForest-ML/data/raw/fire_google_drive


## 11. Zenodo Fire
ดาวน์โหลดไฟล์เสียงที่เผยแพร่กับ record นี้ไว้ใน `data/raw/zenodo_fire/`.

In [45]:
ZENODO_FIRE_DIR = RAW_DIR / 'zenodo_fire'
if has_audio(ZENODO_FIRE_DIR):
    print('[skip] Zenodo Fire: พบไฟล์เสียงแล้ว')
else:
    fire_zip = download_file('https://zenodo.org/records/20132520/files/data.zip?download=1', RAW_DIR / 'archives' / 'zenodo_fire_data.zip')
    extract_zip(fire_zip, ZENODO_FIRE_DIR)


[skip] Zenodo Fire: พบไฟล์เสียงแล้ว


## 12. สรุปจำนวนและขนาดไฟล์ แล้วบันทึก manifest

In [46]:
from datetime import datetime, timezone
import json
def summarize(folder):
    files = [p for p in folder.rglob('*') if p.is_file() and p.name not in {'.DS_Store', '.gitkeep'}]
    return {'path': str(folder.relative_to(PROJECT_ROOT)), 'file_count': len(files), 'total_bytes': sum(p.stat().st_size for p in files)}
DATASEC_SUMMARY_DIR = RAW_DIR / 'datasec_v2'
if not has_audio(DATASEC_SUMMARY_DIR) and has_audio(RAW_DIR / 'datasec_vehicle_v2'):
    DATASEC_SUMMARY_DIR = RAW_DIR / 'datasec_vehicle_v2'
DATASET_DIRS = {
    'esc50': RAW_DIR / 'esc50', 'footstep_set': RAW_DIR / 'footstep_set',
    'datasec_v2': DATASEC_SUMMARY_DIR, 'rfcx_frugalai': RAW_DIR / 'rfcx_frugalai',
    'fsc22': RAW_DIR / 'fsc22', 'fsd50k_v1': RAW_DIR / 'fsd50k_v1',
    'fire_google_drive': RAW_DIR / 'fire_google_drive', 'zenodo_fire': RAW_DIR / 'zenodo_fire',
    'vehicle_sounds': RAW_DIR / 'vehicle_sounds',
}
summary = {name: summarize(path) for name, path in DATASET_DIRS.items()}
manifest = {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'datasets': summary}
manifest_path = PROJECT_ROOT / 'data' / 'manifests' / 'download_manifest.json'
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
for name, info in summary.items(): print(f"{name}: {info['file_count']:,} files, {info['total_bytes'] / (1024**3):.2f} GiB")
print('Manifest:', manifest_path.relative_to(PROJECT_ROOT))


esc50: 2,013 files, 1.42 GiB
footstep_set: 2,129 files, 1.15 GiB
datasec_v2: 5,049 files, 12.95 GiB
rfcx_frugalai: 58 files, 10.64 GiB
fsc22: 2,027 files, 1.67 GiB
fsd50k_v1: 0 files, 0.00 GiB
fire_google_drive: 722 files, 0.64 GiB
zenodo_fire: 48 files, 22.72 GiB
vehicle_sounds: 9,448 files, 7.83 GiB
Manifest: data/manifests/download_manifest.json
